# Week 1 — Exercise 1: Individual Metadata Creation

**Goal:** Apply DataCite and schema.org metadata standards to a publicly available dataset.

**Dataset chosen:** `Ransomware Dataset 2024` — a tabular dataset of PE-file static and dynamic features for malware classification, published on Zenodo under DOI [10.5281/zenodo.13890887](https://doi.org/10.5281/zenodo.13890887).

The dataset contains 21,752 balanced Windows PE samples (10,876 benign and 10,876 malicious) spanning 26 malware families including ransomware strains (WannaCry, Ryuk, REvil, LockBit, …), RATs, trojans, and stealers. Features include DOS/PE header fields, section attributes, and sandbox-derived behavioral counters (registry, network, process, file activity). It supports supervised learning for ransomware detection and multi-class family classification.

This notebook:
1. Inspects the CSV to extract factual metadata (row count, column count, size, schema).
2. Builds **DataCite 4.5** metadata as XML.
3. Builds **schema.org** metadata as JSON-LD with `@type: Dataset`.
4. Writes both files next to the notebook.

## 1. Inspect the dataset

In [2]:
from pathlib import Path
import hashlib
import json
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
DATASET_PATH = NOTEBOOK_DIR / "Dataset" / "Final_Dataset_without_duplicate.csv"

assert DATASET_PATH.exists(), f"Dataset not found at {DATASET_PATH}"

df = pd.read_csv(DATASET_PATH, low_memory=False)
size_bytes = DATASET_PATH.stat().st_size

sha256 = hashlib.sha256()
with DATASET_PATH.open("rb") as fh:
    for chunk in iter(lambda: fh.read(1 << 20), b""):
        sha256.update(chunk)
checksum = sha256.hexdigest()

summary = {
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "size_bytes": size_bytes,
    "size_mb": round(size_bytes / (1024 * 1024), 2),
    "sha256": checksum,
    "class_label_distribution": df["Class"].value_counts().to_dict(),
    "malware_family_count": int(df["Family"].nunique()),
}
print(json.dumps(summary, indent=2))

{
  "rows": 21752,
  "columns": 77,
  "size_bytes": 20677619,
  "size_mb": 19.72,
  "sha256": "dd8347389b98dc4ad9552353262079efab3c600148bcfa1d0c1f94186b27cb8e",
  "class_label_distribution": {
    "Benign": 10876,
    "Malware": 10876
  },
  "malware_family_count": 27
}


In [3]:
print("Columns (" + str(df.shape[1]) + "):")
for col in df.columns:
    print(f"  - {col}: {df[col].dtype}")

Columns (77):
  - md5: str
  - sha1: str
  - file_extension: str
  - EntryPoint: str
  - PEType: str
  - MachineType: str
  - magic_number: str
  - bytes_on_last_page: str
  - pages_in_file: str
  - relocations: str
  - size_of_header: str
  - min_extra_paragraphs: str
  - max_extra_paragraphs: str
  - init_ss_value: str
  - init_sp_value: str
  - init_ip_value: str
  - init_cs_value: str
  - over_lay_number: str
  - oem_identifier: str
  - address_of_ne_header: str
  - Magic: str
  - SizeOfCode: str
  - SizeOfInitializedData: str
  - SizeOfUninitializedData: str
  - AddressOfEntryPoint: str
  - BaseOfCode: str
  - BaseOfData: str
  - ImageBase: str
  - SectionAlignment: str
  - FileAlignment: str
  - OperatingSystemVersion: str
  - ImageVersion: str
  - SizeOfImage: str
  - SizeOfHeaders: str
  - Checksum: str
  - Subsystem: str
  - DllCharacteristics: str
  - SizeofStackReserve: str
  - SizeofStackCommit: str
  - SizeofHeapCommit: str
  - SizeofHeapReserve: str
  - LoaderFlags: str
 

## 2. Shared descriptive facts

Central place for the human-authored metadata. The XML and JSON-LD builders below consume this dictionary so the two files stay consistent.

In [4]:
META = {
    "identifier_doi": "10.5281/zenodo.13890887",
    "landing_page": "https://doi.org/10.5281/zenodo.13890887",
    "title": "Ransomware Dataset 2024",
    "creators": [
        {"name": "Hussain, Amjad", "orcid": None, "affiliation": "Air University"},
    ],
    "publisher": "Zenodo",
    "publication_year": "2024",
    "resource_type": "Dataset",
    "description": (
        "Balanced tabular dataset of 21,752 Windows PE file samples (10,876 benign "
        "and 10,876 malicious) spanning 26 malware families including ransomware "
        "strains (WannaCry, Ryuk, REvil, LockBit, Cerber, Dharma, Gandcrab, …), RATs, "
        "trojans, and stealers. Each row captures DOS/PE header fields (MZ/PE magic, "
        "section counts, image sizes, entry point, subsystem, DLL characteristics), "
        ".text and .rdata section attributes, and sandbox-derived behavioral counters "
        "(registry read/write/delete, network DNS/HTTP/connections, process and file "
        "activity, DLL calls, API count). Target columns are Class (Benign/Malware), "
        "Category, and Family. Supports supervised learning for ransomware detection "
        "and multi-class malware family classification."
    ),
    "subjects": [
        "ransomware",
        "malware detection",
        "malware classification",
        "PE file analysis",
        "cybersecurity",
        "machine learning",
        "dynamic analysis",
        "static analysis",
    ],
    "keywords": [
        "ransomware dataset",
        "malware dataset",
        "ransomware dataset 2024",
        "malware dataset 2024",
        "PE features",
        "cybersecurity",
    ],
    "language": "en",
    "license_name": "Creative Commons Attribution 4.0 International",
    "license_spdx": "CC-BY-4.0",
    "license_url": "https://creativecommons.org/licenses/by/4.0/",
    "format": "text/csv",
    "related_publication_doi": None,
    "related_publication_citation": None,
    "funder": None,
    "version": "1.0",
    "rows": summary["rows"],
    "columns": summary["columns"],
    "size_bytes": summary["size_bytes"],
    "sha256": summary["sha256"],
}
print(json.dumps({k: v for k, v in META.items() if k != "creators"}, indent=2))

{
  "identifier_doi": "10.5281/zenodo.13890887",
  "landing_page": "https://doi.org/10.5281/zenodo.13890887",
  "title": "Ransomware Dataset 2024",
  "publisher": "Zenodo",
  "publication_year": "2024",
  "resource_type": "Dataset",
  "description": "Balanced tabular dataset of 21,752 Windows PE file samples (10,876 benign and 10,876 malicious) spanning 26 malware families including ransomware strains (WannaCry, Ryuk, REvil, LockBit, Cerber, Dharma, Gandcrab, \u2026), RATs, trojans, and stealers. Each row captures DOS/PE header fields (MZ/PE magic, section counts, image sizes, entry point, subsystem, DLL characteristics), .text and .rdata section attributes, and sandbox-derived behavioral counters (registry read/write/delete, network DNS/HTTP/connections, process and file activity, DLL calls, API count). Target columns are Class (Benign/Malware), Category, and Family. Supports supervised learning for ransomware detection and multi-class malware family classification.",
  "subjects": [


## 3. Build DataCite 4.5 XML

Follows the DataCite Metadata Schema 4.5 (`http://datacite.org/schema/kernel-4`). Mandatory properties: Identifier, Creator, Title, Publisher, PublicationYear, ResourceType.

In [5]:
from lxml import etree

NS = "http://datacite.org/schema/kernel-4"
XSI = "http://www.w3.org/2001/XMLSchema-instance"
SCHEMA_LOC = (
    "http://datacite.org/schema/kernel-4 "
    "http://schema.datacite.org/meta/kernel-4.5/metadata.xsd"
)

nsmap = {None: NS, "xsi": XSI}
resource = etree.Element(f"{{{NS}}}resource", nsmap=nsmap)
resource.set(f"{{{XSI}}}schemaLocation", SCHEMA_LOC)

identifier = etree.SubElement(resource, "identifier", identifierType="DOI")
identifier.text = META["identifier_doi"]

creators_el = etree.SubElement(resource, "creators")
for c in META["creators"]:
    creator = etree.SubElement(creators_el, "creator")
    name = etree.SubElement(creator, "creatorName")
    name.text = c["name"]
    if ", " in c["name"]:
        family, _, given = c["name"].partition(", ")
        name.set("nameType", "Personal")
        etree.SubElement(creator, "familyName").text = family
        etree.SubElement(creator, "givenName").text = given
    if c.get("orcid"):
        nid = etree.SubElement(
            creator, "nameIdentifier",
            nameIdentifierScheme="ORCID",
            schemeURI="https://orcid.org/",
        )
        nid.text = f"https://orcid.org/{c['orcid']}"
    if c.get("affiliation"):
        etree.SubElement(creator, "affiliation").text = c["affiliation"]

titles = etree.SubElement(resource, "titles")
etree.SubElement(titles, "title", attrib={"{http://www.w3.org/XML/1998/namespace}lang": META["language"]}).text = META["title"]

etree.SubElement(resource, "publisher").text = META["publisher"]
etree.SubElement(resource, "publicationYear").text = META["publication_year"]

subjects_el = etree.SubElement(resource, "subjects")
for s in META["subjects"]:
    etree.SubElement(subjects_el, "subject").text = s

etree.SubElement(
    resource, "language",
).text = META["language"]

resource_type = etree.SubElement(
    resource, "resourceType", resourceTypeGeneral="Dataset",
)
resource_type.text = "Tabular dataset (CSV)"

sizes = etree.SubElement(resource, "sizes")
etree.SubElement(sizes, "size").text = f"{META['size_bytes']} bytes"
etree.SubElement(sizes, "size").text = f"{META['rows']} rows"
etree.SubElement(sizes, "size").text = f"{META['columns']} columns"

formats = etree.SubElement(resource, "formats")
etree.SubElement(formats, "format").text = META["format"]

etree.SubElement(resource, "version").text = META["version"]

rights_list = etree.SubElement(resource, "rightsList")
rights = etree.SubElement(
    rights_list, "rights",
    rightsURI=META["license_url"],
    rightsIdentifier=META["license_spdx"],
    rightsIdentifierScheme="SPDX",
)
rights.text = META["license_name"]

descriptions = etree.SubElement(resource, "descriptions")
etree.SubElement(
    descriptions, "description", descriptionType="Abstract",
).text = META["description"]

if META.get("related_publication_doi"):
    related = etree.SubElement(resource, "relatedIdentifiers")
    etree.SubElement(
        related, "relatedIdentifier",
        relatedIdentifierType="DOI",
        relationType="IsDocumentedBy",
    ).text = META["related_publication_doi"]

xml_bytes = etree.tostring(
    resource, pretty_print=True, xml_declaration=True, encoding="UTF-8",
)
print(xml_bytes.decode("utf-8"))

<?xml version='1.0' encoding='UTF-8'?>
<resource xmlns="http://datacite.org/schema/kernel-4" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.5/metadata.xsd">
  <identifier identifierType="DOI">10.5281/zenodo.13890887</identifier>
  <creators>
    <creator>
      <creatorName nameType="Personal">Hussain, Amjad</creatorName>
      <familyName>Hussain</familyName>
      <givenName>Amjad</givenName>
      <affiliation>Air University</affiliation>
    </creator>
  </creators>
  <titles>
    <title xml:lang="en">Ransomware Dataset 2024</title>
  </titles>
  <publisher>Zenodo</publisher>
  <publicationYear>2024</publicationYear>
  <subjects>
    <subject>ransomware</subject>
    <subject>malware detection</subject>
    <subject>malware classification</subject>
    <subject>PE file analysis</subject>
    <subject>cybersecurity</subject>
    <subject>machine learning</subject>
    <subject>dyn

In [6]:
xml_path = NOTEBOOK_DIR / "Dataset_datacite.xml"
xml_path.write_bytes(xml_bytes)
print(f"Wrote {xml_path} ({xml_path.stat().st_size} bytes)")

Wrote e:\Master's Notes\2nd semester\Data Management\Data Management Assignments\Data-Management\Week_1_Exercise_1_Individual_Metadata_Creation\Dataset_datacite.xml (2423 bytes)


## 4. Build schema.org JSON-LD

Uses `@type: Dataset` per https://schema.org/Dataset. Includes `distribution` (DataDownload) and `variableMeasured` entries derived from the CSV columns, which makes the metadata richer than the DataCite XML while staying consistent with it.

In [7]:
variable_measured = [{"@type": "PropertyValue", "name": col} for col in df.columns]

schema_org = {
    "@context": "https://schema.org/",
    "@type": "Dataset",
    "@id": META["landing_page"],
    "identifier": [
        {
            "@type": "PropertyValue",
            "propertyID": "DOI",
            "value": META["identifier_doi"],
            "url": META["landing_page"],
        }
    ],
    "name": META["title"],
    "description": META["description"],
    "url": META["landing_page"],
    "version": META["version"],
    "datePublished": META["publication_year"],
    "inLanguage": META["language"],
    "license": META["license_url"],
    "keywords": META["keywords"],
    "creator": [
        {
            "@type": "Person",
            "name": c["name"],
            **({"identifier": f"https://orcid.org/{c['orcid']}"} if c.get("orcid") else {}),
            **({"affiliation": {"@type": "Organization", "name": c["affiliation"]}} if c.get("affiliation") else {}),
        }
        for c in META["creators"]
    ],
    "publisher": {
        "@type": "Organization",
        "name": META["publisher"],
        "url": "https://zenodo.org/",
    },
    "isAccessibleForFree": True,
    "encodingFormat": META["format"],
    "distribution": [
        {
            "@type": "DataDownload",
            "encodingFormat": META["format"],
            "contentUrl": META["landing_page"],
            "contentSize": f"{META['size_bytes']} B",
            "sha256": META["sha256"],
        }
    ],
    "variableMeasured": variable_measured,
    "size": {
        "@type": "QuantitativeValue",
        "unitText": "rows",
        "value": META["rows"],
    },
}

if META.get("related_publication_citation"):
    schema_org["citation"] = META["related_publication_citation"]
if META.get("related_publication_doi"):
    schema_org["isBasedOn"] = f"https://doi.org/{META['related_publication_doi']}"

print(json.dumps(schema_org, indent=2)[:2000] + "\n...")

{
  "@context": "https://schema.org/",
  "@type": "Dataset",
  "@id": "https://doi.org/10.5281/zenodo.13890887",
  "identifier": [
    {
      "@type": "PropertyValue",
      "propertyID": "DOI",
      "value": "10.5281/zenodo.13890887",
      "url": "https://doi.org/10.5281/zenodo.13890887"
    }
  ],
  "name": "Ransomware Dataset 2024",
  "description": "Balanced tabular dataset of 21,752 Windows PE file samples (10,876 benign and 10,876 malicious) spanning 26 malware families including ransomware strains (WannaCry, Ryuk, REvil, LockBit, Cerber, Dharma, Gandcrab, \u2026), RATs, trojans, and stealers. Each row captures DOS/PE header fields (MZ/PE magic, section counts, image sizes, entry point, subsystem, DLL characteristics), .text and .rdata section attributes, and sandbox-derived behavioral counters (registry read/write/delete, network DNS/HTTP/connections, process and file activity, DLL calls, API count). Target columns are Class (Benign/Malware), Category, and Family. Supports su

In [ ]:
jsonld_path = NOTEBOOK_DIR / "Final_Dataset_without_duplicate_schemaorg.json"
jsonld_path.write_text(json.dumps(schema_org, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {jsonld_path} ({jsonld_path.stat().st_size} bytes)")

Wrote e:\Master's Notes\2nd semester\Data Management\Data Management Assignments\Data-Management\Week_1_Exercise_1_Individual_Metadata_Creation\Dataset_schema.org.json (8475 bytes)


## 5. Verify the outputs

In [9]:
parsed_xml = etree.fromstring(xml_path.read_bytes())
assert parsed_xml.tag.endswith("resource"), "Root element must be <resource>"

parsed_jsonld = json.loads(jsonld_path.read_text(encoding="utf-8"))
assert parsed_jsonld["@type"] == "Dataset"
assert parsed_jsonld["@context"].startswith("https://schema.org")

print("DataCite XML — well-formed ✅")
print("schema.org JSON-LD — parses and declares @type: Dataset ✅")

DataCite XML — well-formed ✅
schema.org JSON-LD — parses and declares @type: Dataset ✅
